In [2]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import timedelta
import random
import yfinance as yf  
from tqdm import tqdm
import os

## Method 1: 
Get ticket details

In [4]:
def get_ticker_details(ticker_symbol):
    """Fetches financial data for a single ticker."""
    time.sleep(0.5) 
    try:
        ticker_obj = yf.Ticker(ticker_symbol)
        info = ticker_obj.info
        # If info is empty or ticker doesn't exist, yfinance often returns an empty dict
        if not info:
            return [None, None, "Unknown", None, None, "N/A"]
            
        return [
            info.get('currentPrice') or info.get('regularMarketPrice'),
            info.get('marketCap'),
            info.get('currency', 'Unknown'),
            info.get('trailingEps'),
            info.get('trailingPE'),
            info.get('sector', 'N/A')
        ]
    except Exception:
        return [None, None, "Unknown", None, None, "N/A"]



## Methods 2 and 3: 
Load transactions from parquet files, use method 1 to get details and then clean up both the detail ticker info AND the resulting transaction file

In [ ]:
def update_master_with_new_tickers(df_master, parquet_path):
    """Identifies new tickers, fetches data, and appends to master."""
    # 1. Load transactions
    df_tx_full = pd.read_parquet(parquet_path, engine='fastparquet')
    
    # Extract unique tickers for checking
    unique_tx_tickers = (df_tx_full['issuerSign'] + '.OL').unique()

    # 2. Compare and find missing tickers
    existing_tickers = set(df_master['Tickers'])
    new_tickers = [t for t in unique_tx_tickers if t not in existing_tickers]

    if new_tickers:
        print(f"Found {len(new_tickers)} new tickers. Fetching data...")
        new_rows = []
        for ticker in new_tickers:
            print(f"Processing: {ticker}")
            details = get_ticker_details(ticker)
            new_rows.append([ticker] + details)
        
        df_new_entries = pd.DataFrame(new_rows, columns=df_master.columns)
        df_master = pd.concat([df_master, df_new_entries], ignore_index=True)
    else:
        print("No new tickers to add.")

    # 3. CLEANING LOGIC
    # Remove rows where Currency is Unknown OR Sector is N/A (or NaN)
    # We use .copy() to avoid SettingWithCopy warnings later
    df_master_cleaned = df_master[
        (df_master['Currency'] != 'Unknown') & 
        (df_master['Sector'].notna()) & 
        (df_master['Sector'] != 'N/A')
    ].copy()

    # 4. FILTER TRANSACTIONS
    # Only keep transactions for tickers that survived the cleaning
    valid_tickers = set(df_master_cleaned['Tickers'])
    # Remove the '.OL' suffix to match the original 'issuerSign' in the parquet
    valid_issuer_signs = {t.replace('.OL', '') for t in valid_tickers}
    
    df_tx_cleaned = df_tx_full[df_tx_full['issuerSign'].isin(valid_issuer_signs)].copy()

    return df_master_cleaned, df_tx_cleaned



In [ ]:
# --- EXECUTION ---

MASTER_FILE = 'oslo_stock_data.csv'
PARQUET_FOLDER = 'transaction_data/'
CLEANED_TX_FILE = 'cleaned_transactions.csv'

# Load (or create) the Master DataFrame
if os.path.exists(MASTER_FILE):
    master_df = pd.read_csv(MASTER_FILE)
else:
    cols = ['Tickers', 'Price', 'Market Cap', 'Currency', 'EPS', 'PE Ratio', 'Sector']
    master_df = pd.DataFrame(columns=cols)

# Run the update and filtering
master_df, cleaned_tx_df = update_master_with_new_tickers(master_df, PARQUET_FOLDER)

# Save the Results
master_df.to_csv(MASTER_FILE, index=False, encoding='utf-8-sig')
cleaned_tx_df.to_csv(CLEANED_TX_FILE, index=False, encoding='utf-8-sig')

print(f"Workflow complete.")
print(f"Master file saved to: {MASTER_FILE}")
print(f"Cleaned transactions saved to: {CLEANED_TX_FILE}")